### Axisymmetric Pipe Expansion under Internal Pressure (R–Z)

This notebook models a **thick-walled cylindrical pipe** in 2D axisymmetry (R–Z) under internal pressure.  
Linear elasticity (small strains) is assumed. The solver computes **displacements** $u_r$, **strains** $\varepsilon_{rr}, \varepsilon_{\theta\theta}, \varepsilon_{zz}$, **stresses** $\sigma_{rr}, \sigma_{\theta\theta}, \sigma_{zz}$, and the **von Mises equivalent stress** $\sigma_{\mathrm{vm}}$.

**Loading sweep** (Pa):  
`[0, 1e5, 2e5, 5e5, 1e6, 2e6]`  *(i.e., 0 → 2 MPa)*

**Boundary conditions**
- Internal radius: uniform **internal pressure** $p_i$.
- External radius: **free surface** (zero traction).
- Axial direction: **plane-strain** assumption $(\varepsilon_{zz}=0)$.
- Axisymmetric operators include the **$1/r$** terms.

Outputs include field maps for $u_r$, stresses, and $\sigma_{\mathrm{vm}}$, enabling direct comparison with **Lamé’s thick-cylinder** formulas and assessment of pressure scaling.

**NOTE** Discrepancies arise between sequential and parallel executions due to the absence of boundary anchoring. Consequently, different spurious modes are allowed to develop in each configuration.

In [ ]:
from trustutils import run
run.reset()
run.initBuildDirectory()
h = 0.1
z0 = 0
ri = 0.24
t_w = 2e-3
re = ri + t_w

E = 160e9
nu = 0.31
pressures = [0, 1e5, 2e5, 5e5, 10e5, 20e5]
for p in pressures:
    run.addCaseFromTemplate("jdd.data", f"case{int(p/1e5)}", {"p": p, "E" : E, "nu" : nu, "t_w": t_w, "ri": ri, "re": re})
run.runCases()
run.tablePerf()

In [ ]:
from trustutils import plot
import numpy as np
import matplotlib.lines as mlines

# ------------------------
# Réglages / formules
# ------------------------
PLANE_MODE = "stress"   # "stress"  -> plan-contraintes (σz=0)
                        # "strain"  -> plan-déformations (εz=0)

def lamé_AB(pi, po=0.0, ri=None, re=None):
    """Constantes de Lamé pour cylindre épais soumis à pi (interne), po (externe)."""
    if ri is None or re is None or ri <= 0 or re <= ri:
        raise ValueError("ri/re invalides.")
    A = (pi*ri*ri - po*re*re) / (re*re - ri*ri)
    B = (ri*ri * re*re * (pi - po)) / (re*re - ri*ri)
    return A, B

def fields_r(r, pi, E, nu, ri, re, plane_mode="stress"):
    """Retourne u_r, (eps_r, eps_t, eps_z), (sig_r, sig_t, sig_z), vm au rayon r."""
    A, B = lamé_AB(pi, 0.0, ri, re)

    # Contraintes radiale/circonférentielle (formules classiques)
    sig_r = A - B / (r*r)
    sig_t = A + B / (r*r)

    if plane_mode == "stress":  # σz = 0
        sig_z = 0.0
        eps_r = (1.0/E)*((1-nu)*A - (1+nu)*B/(r*r))
        eps_t = (1.0/E)*((1-nu)*A + (1+nu)*B/(r*r))
        eps_z = -(2*nu/E)*A
        u_r   = (1.0/E)*((1-nu)*A*r + (1+nu)*B/r)
    elif plane_mode == "strain":  # εz = 0 -> σz = ν(σr+σt) = 2νA
        sig_z = 2.0*nu*A
        eps_r = (1.0/E)*((1-nu-2*nu*nu)*A - (1+nu)*B/(r*r))
        eps_t = (1.0/E)*((1-nu-2*nu*nu)*A + (1+nu)*B/(r*r))
        eps_z = 0.0
        u_r   = (1.0/E)*((1-nu-2*nu*nu)*A*r + (1+nu)*B/r)
    else:
        raise ValueError("plane_mode doit être 'stress' ou 'strain'.")

    # Von Mises 3D (valable dans les deux modes)
    vm = np.sqrt(0.5*((sig_t - sig_r)**2 + (sig_r - sig_z)**2 + (sig_z - sig_t)**2))
    return u_r, (eps_r, eps_t, eps_z), (sig_r, sig_t, sig_z), vm

def case_dir(p):
    """Dossier du cas en fonction de la pression (par pas de 1 bar)."""
    return f"{run.BUILD_DIRECTORY}/case{int(p/1e5)}"

def label_bar(p):
    """Label pression en bar."""
    return f"{int(p/1e5)} b"

# Palette circulaire (autant de couleurs que nécessaire)
base_colors = ["b", "g", "r", "c", "m", "y", "k"]
def color_i(i):
    return base_colors[i % len(base_colors)]

# ------------------------
# Légende custom
# ------------------------
def apply_custom_legend(graph, pressures):
    """
    Légende custom :
      - couleur : pression
      - 'x' : TRUST
      - ligne pointillée : formule analytique
      Placée à l'extérieur, à droite.
    """
    # Récupérer l'axe matplotlib sous-jacent
    try:
        ax = graph.ax              # suivant la version de trustutils
    except AttributeError:
        ax = graph.fig.axes[0]     # fallback classique

    # Récupérer la figure
    try:
        fig = graph.fig
    except AttributeError:
        fig = ax.figure

    # Laisser de la place à droite pour la légende
    fig.subplots_adjust(right=0.75)

    # Supprimer une éventuelle légende existante
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()

    # 1) une entrée par pression -> encode la COULEUR seulement
    handles_pressures = []
    for i, p in enumerate(pressures):
        h = mlines.Line2D(
            [], [],
            color=color_i(i),
            linestyle="-",
            marker="",
            label=label_bar(p)      # "1 b", "2 b", ...
        )
        handles_pressures.append(h)

    # 2) une entrée pour TRUST (symbole x)
    handle_trust = mlines.Line2D(
        [], [],
        color="black",
        linestyle="",
        marker="x",
        label="TRUST"
    )

    # 3) une entrée pour la formule (ligne pointillée)
    handle_formula = mlines.Line2D(
        [], [],
        color="black",
        linestyle="--",
        marker="",
        label="Formule analytique"
    )

    handles = handles_pressures + [handle_trust, handle_formula]

    ax.legend(
        handles=handles,
        loc="center left",         # ancrage vertical au centre
        bbox_to_anchor=(1.02, 0.5),# juste à droite de l'axe
        borderaxespad=0.0,
        title="Légende"
    )

# ------------------------
# Échantillonnage radial (sur toute l'épaisseur)
# ------------------------
x = np.linspace(ri, re, 200)

# ------------------------
# 1) Résidus solveur
# ------------------------
fig_res = plot.Graph("Solver residuals")
for p_idx, p in enumerate(pressures):
    fig_res.addResidu(
        f"{case_dir(p)}/jdd.dt_ev",
        label=f"{label_bar(p)}"
    )
fig_res.scale("linear", "log")
# Ici on garde la légende auto par pression, c'est cohérent.

# ------------------------
# 2) Déplacement radial u_r(r)
# ------------------------
fig_ur = plot.Graph("Radial displacement $u_r(r)$")
for p_idx, p in enumerate(pressures):
    # TRUST
    fig_ur.addSegment(
        f"{case_dir(p)}/jdd_DX.son",
        compo=0,
        marker="x",
        color=color_i(p_idx)
    )
    # Formule fermée
    ur = [fields_r(r, p, E, nu, ri, re, PLANE_MODE)[0] for r in x]
    fig_ur.add(
        x, ur,
        marker="--",                    # style "formule"
        color=color_i(p_idx)
    )

apply_custom_legend(fig_ur, pressures)

# ------------------------
# 3) Von Mises σ_vm(r)
# ------------------------
fig_vm = plot.Graph("Von Mises $\\sigma_{\\mathrm{vm}}(r)$")
for p_idx, p in enumerate(pressures):
    # TRUST
    fig_vm.addSegment(
        f"{case_dir(p)}/jdd_VM.son",
        marker="x",
        label="_nolegend_",
        color=color_i(p_idx)
    )
    # Formule
    vm = [fields_r(r, p, E, nu, ri, re, PLANE_MODE)[3] for r in x]
    fig_vm.add(
        x, vm,
        marker="--",
        label="_nolegend_",
        color=color_i(p_idx)
    )

apply_custom_legend(fig_vm, pressures)

# ------------------------
# 4) Contraintes σ_r, σ_θ, σ_z
# ------------------------
fig_sr = plot.Graph("$\\sigma_r(r)$")
fig_st = plot.Graph("$\\sigma_\\theta(r)$")
fig_sz = plot.Graph("$\\sigma_z(r)$")

for p_idx, p in enumerate(pressures):
    # TRUST
    fig_sr.addSegment(
        f"{case_dir(p)}/jdd_SIGMA.son",
        compo=0,
        marker="x",
        label="_nolegend_",
        color=color_i(p_idx)
    )
    fig_st.addSegment(
        f"{case_dir(p)}/jdd_SIGMA.son",
        compo=2,
        marker="x",
        label="_nolegend_",
        color=color_i(p_idx)
    )
    fig_sz.addSegment(
        f"{case_dir(p)}/jdd_SIGMA.son",
        compo=1,
        marker="x",
        label="_nolegend_",
        color=color_i(p_idx)
    )

    # Formules
    sig_r = []
    sig_t = []
    sig_z = []
    for r in x:
        _, _, (sr, st, sz), _ = fields_r(r, p, E, nu, ri, re, PLANE_MODE)
        sig_r.append(sr)
        sig_t.append(st)
        sig_z.append(sz)

    fig_sr.add(
        x, sig_r,
        marker="--",
        label="_nolegend_",
        color=color_i(p_idx)
    )
    fig_st.add(
        x, sig_t,
        marker="--",
        label="_nolegend_",
        color=color_i(p_idx)
    )
    fig_sz.add(
        x, sig_z,
        marker="--",
        label="_nolegend_",
        color=color_i(p_idx)
    )

apply_custom_legend(fig_sr, pressures)
apply_custom_legend(fig_st, pressures)
apply_custom_legend(fig_sz, pressures)

# ------------------------
# 5) Déformations ε_r, ε_θ, ε_z
# ------------------------
fig_er = plot.Graph("$\\varepsilon_r(r)$")
fig_et = plot.Graph("$\\varepsilon_\\theta(r)$")
fig_ez = plot.Graph("$\\varepsilon_z(r)$")

for p_idx, p in enumerate(pressures):
    # TRUST
    fig_er.addSegment(
        f"{case_dir(p)}/jdd_EPSILON.son",
        compo=0,
        marker="x",
        label="_nolegend_",
        color=color_i(p_idx)
    )
    fig_et.addSegment(
        f"{case_dir(p)}/jdd_EPSILON.son",
        compo=2,
        marker="x",
        label="_nolegend_",
        color=color_i(p_idx)
    )
    fig_ez.addSegment(
        f"{case_dir(p)}/jdd_EPSILON.son",
        compo=1,
        marker="x",
        label="_nolegend_",
        color=color_i(p_idx)
    )

    # Formules
    eps_r = []
    eps_t = []
    eps_z = []
    for r in x:
        _, (er, et, ez), _, _ = fields_r(r, p, E, nu, ri, re, PLANE_MODE)
        eps_r.append(er)
        eps_t.append(et)
        eps_z.append(ez)

    fig_er.add(
        x, eps_r,
        marker="--",
        label="_nolegend_",
        color=color_i(p_idx)
    )
    fig_et.add(
        x, eps_t,
        marker="--",
        label="_nolegend_",
        color=color_i(p_idx)
    )
    fig_ez.add(
        x, eps_z,
        marker="--",
        label="_nolegend_",
        color=color_i(p_idx)
    )

apply_custom_legend(fig_er, pressures)
apply_custom_legend(fig_et, pressures)
apply_custom_legend(fig_ez, pressures)

# ------------------------
# 2bis) u_r et Von Mises côte à côte, légende commune
# ------------------------
import matplotlib.pyplot as plt

try:
    # Récupérer les axes sources depuis les Graph TRUST
    try:
        ax_ur_src = fig_ur.ax
    except AttributeError:
        ax_ur_src = fig_ur.fig.axes[0]

    try:
        ax_vm_src = fig_vm.ax
    except AttributeError:
        ax_vm_src = fig_vm.fig.axes[0]

    # Nouvelle figure avec deux sous-graphiques côte à côte
    fig_uvm, (ax_ur, ax_vm) = plt.subplots(1, 2, figsize=(10, 4), sharex=False)

    # Copier les courbes de u_r
    for line in ax_ur_src.get_lines():
        ax_ur.plot(
            line.get_xdata(), line.get_ydata(),
            linestyle=line.get_linestyle(),
            marker=line.get_marker(),
            color=line.get_color()
        )

    # Copier les courbes de Von Mises
    for line in ax_vm_src.get_lines():
        ax_vm.plot(
            line.get_xdata(), line.get_ydata(),
            linestyle=line.get_linestyle(),
            marker=line.get_marker(),
            color=line.get_color()
        )

    # Titres / labels
    ax_ur.set_title(r"Radial displacement $u_r(r)$")
    ax_ur.set_xlabel(r"$r$")
    ax_ur.set_ylabel(r"$u_r$")

    ax_vm.set_title(r"Von Mises $\sigma_{\mathrm{vm}}(r)$")
    ax_vm.set_xlabel(r"$r$")
    ax_vm.set_ylabel(r"$\sigma_{\mathrm{vm}}$")

    # Légende commune (même logique que apply_custom_legend)
    handles_pressures = []
    for i, p in enumerate(pressures):
        h = mlines.Line2D(
            [], [],
            color=color_i(i),
            linestyle="-",
            marker="",
            label=label_bar(p)
        )
        handles_pressures.append(h)

    handle_trust = mlines.Line2D(
        [], [],
        color="black",
        linestyle="",
        marker="x",
        label="TRUST"
    )

    handle_formula = mlines.Line2D(
        [], [],
        color="black",
        linestyle="--",
        marker="",
        label="Formule analytique"
    )

    handles = handles_pressures + [handle_trust, handle_formula]

    # On laisse de la place à droite pour la légende commune
    # fig_uvm.subplots_adjust(right=0.8)
    fig_uvm.legend(
        handles=handles,
        loc="center left",
        bbox_to_anchor=(0.95, 0.5),
        borderaxespad=0.0,
    )

except Exception as e:
    print("Impossible de construire la figure combinée u_r / sigma_vm :", e)